# 🧠 情绪价值模型微调 - Colab 版

微调 Qwen 0.6B 模型，使其具备提供情绪价值的能力。

## 使用方法
1. 点击 `Runtime` → `Change runtime type` → 选择 `T4 GPU` 或 `A100 GPU`
2. 按顺序运行下方所有代码块
3. 训练完成后，下载模型或直接测试

In [ ]:
# @title 配置参数
# @markdown 在此处修改训练参数

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # @param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/emotional_model"  # @param {type:"string"}
EPOCHS = 3  # @param {type:"integer"}
BATCH_SIZE = 2  # @param {type:"integer"}
LEARNING_RATE = 2e-4  # @param {type:"number"}
MAX_LENGTH = 2048  # @param {type:"integer"}
LORA_R = 8  # @param {type:"integer"}
LORA_ALPHA = 16  # @param {type:"integer"}
MAX_SAMPLES = 50000  # @param {type:"integer"}

# 是否连接 Google Drive
USE_DRIVE = True  # @param {type:"boolean"}

print(f"模型: {MODEL_NAME}")
print(f"输出: {OUTPUT_DIR}")
print(f"轮数: {EPOCHS}, 批大小: {BATCH_SIZE}, 学习率: {LEARNING_RATE}")

In [ ]:
# @title 第一步：连接 Google Drive (可选)
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive 已连接")
else:
    print("跳过 Google Drive 连接")

In [ ]:
# @title 第二步：安装依赖
!pip install -q torch transformers accelerate peft datasets bitsandbytes einops tqdm pandas
print("依赖安装完成")

In [ ]:
# @title 第三步：下载开源情绪对话数据集
import os
import json
from datasets import load_dataset, Dataset

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# 下载 EmoChat 数据集
print("下载 EmoChat 数据集...")
try:
    emo_dataset = load_dataset("miemie/EmoChat", trust_remote_code=True)
    print(f"EmoChat 下载成功")
except Exception as e:
    print(f"EmoChat 下载失败: {e}")
    emo_dataset = None

# 下载 Emotional_QA 数据集
print("\n下载 Emotional_QA 数据集...")
try:
    emo_qa = load_dataset("siliconflow/Emotional_QA", trust_remote_code=True)
    print(f"Emotional_QA 下载成功")
except Exception as e:
    print(f"Emotional_QA 下载失败: {e}")
    emo_qa = None

In [ ]:
# @title 第四步：构建 SFT 训练数据
import os
import json

def convert_to_sft(dataset, max_samples=MAX_SAMPLES):
    samples = []
    
    if dataset is None:
        return samples
    
    if isinstance(dataset, dict):
        for split_name in list(dataset.keys())[:1]:
            for item in dataset[split_name]:
                sample = convert_item(item)
                if sample:
                    samples.extend(sample)
                if len(samples) >= max_samples:
                    break
    else:
        for item in dataset:
            sample = convert_item(item)
            if sample:
                samples.extend(sample)
            if len(samples) >= max_samples:
                break
    
    return samples

def convert_item(item):
    results = []
    
    if "instruction" in item and "output" in item:
        user_input = item.get("input", "") or item.get("instruction", "")
        assistant_output = item["output"]
        if user_input and assistant_output:
            results.append({
                "conversations": [
                    {"from": "human", "value": user_input},
                    {"from": "gpt", "value": assistant_output}
                ]
            })
    elif "question" in item and "answer" in item:
        results.append({
            "conversations": [
                {"from": "human", "value": item["question"]},
                {"from": "gpt", "value": item["answer"]}
            ]
        })
    else:
        user_key = None
        assistant_key = None
        for key in ["user", "query", "question", "input", "prompt"]:
            if key in item:
                user_key = key
                break
        for key in ["assistant", "reply", "answer", "response", "output"]:
            if key in item:
                assistant_key = key
                break
        if user_key and assistant_key:
            results.append({
                "conversations": [
                    {"from": "human", "value": str(item[user_key])},
                    {"from": "gpt", "value": str(item[assistant_key])}
                ]
            })
    
    return results

# 转换数据
print("转换数据中...")
sft_samples = []

if 'emo_dataset' in dir() and emo_dataset is not None:
    sft_samples.extend(convert_to_sft(emo_dataset))
    print(f"EmoChat 样本数: {len(sft_samples)}")

if 'emo_qa' in dir() and emo_qa is not None:
    qa_samples = convert_to_sft(emo_qa)
    sft_samples.extend(qa_samples)
    print(f"Emotional_QA 样本数: {len(qa_samples)}")

print(f"\n总样本数: {len(sft_samples)}")

In [ ]:
# @title 第五步：添加高质量自定义情绪样本
custom_samples = [
    {"conversations": [
        {"from": "human", "value": "我今天心情不好，工作压力太大了"},
        {"from": "gpt", "value": "我能感受到你现在承受着很大的压力，真的很不容易。工作压力大的时候，情绪低落是很正常的，你不用强迫自己马上好起来。\n\n或许我们可以一起想想，是什么让你感到特别疲惫？是任务太多，还是对自己的要求太高了呢？"}
    ]},
    {"conversations": [
        {"from": "human", "value": "我觉得自己什么都做不好，没有自信"},
        {"from": "gpt", "value": "你会这样想，说明你对自己有很高的期望，这本身就说明你是一个有追求的人。很多时候，我们对自己最苛刻，却对别人很宽容。\n\n能不能告诉我，最近发生了什么让你有这样的感受？也许我们可以一起梳理一下，看看事情是否真的像你想的那么糟糕。"}
    ]},
    {"conversations": [
        {"from": "human", "value": "我失恋了，很难过"},
        {"from": "gpt", "value": "失恋真的是一件非常痛苦的事情，允许自己难过、哭泣都是很正常的。一段感情的结束不代表你不值得被爱，只是双方可能不再适合了。\n\n你愿意说说你们之间发生了什么吗？倾诉出来可能会让你感觉好一些。无论如何，我都在这里陪着你。"}
    ]},
    {"conversations": [
        {"from": "human", "value": "我要放弃了，感觉坚持不下去了"},
        {"from": "gpt", "value": "你能走到今天，已经证明了你的坚持和勇气。想要放弃的念头恰恰说明你一直在努力着，这本身就很了不起。\n\n可以告诉我是什么让你觉得特别累吗？有时候我们不需要一个解决方案，只是需要有人理解我们的感受。你愿意给自己再多一点时间吗？"}
    ]},
    {"conversations": [
        {"from": "human", "value": "我总是担心未来，焦虑到睡不着"},
        {"from": "gpt", "value": "对未来感到焦虑是很正常的反应，尤其是当我们面临不确定的时候。你试着在脑海中预演各种可能的结果，但这反而增加了心理负担。\n\n或许我们可以尝试一个方法：把你担心的事情写下来，然后问自己，'这些事情真的都在我的控制范围内吗？'很多时候，我们焦虑的大部分事情其实并不一定会发生。"}
    ]},
    {"conversations": [
        {"from": "human", "value": "我觉得没有人理解我"},
        {"from": "gpt", "value": "那种觉得不被理解的孤独感真的很让人难过。我想让你知道，在这里，我会认真地听你说，努力去理解你的感受。\n\n你愿意告诉我，你最想被理解的是什么吗？有时候，我们不需要别人的建议，只是需要有人愿意倾听。我就在这里。"}
    ]},
    {"conversations": [
        {"from": "human", "value": "我讨厌现在的生活状态"},
        {"from": "gpt", "value": "对当前的生活感到不满意，其实是内心在呼唤改变，这是一件好事！它说明你还没有放弃对更好生活的追求。\n\n能不能告诉我，具体是哪些方面让你感到不满？是工作、人际关系，还是个人成长？当我们能够清楚地表达出来的时候，改变的第一步就已经迈出了。"}
    ]},
    {"conversations": [
        {"from": "human", "value": "我总是对自己要求太高，活得好累"},
        {"from": "gpt", "value": "能够意识到自己对自己要求太高，这是一种很好的自我觉察。你值得被温柔对待，特别是来自你自己的温柔。\n\n试试对自己说：'我已经尽力了，我值得休息和被爱。'允许自己不完美，允许自己偶尔偷懒，这不是懒惰，而是自我关怀。"}
    ]},
    {"conversations": [
        {"from": "human", "value": "我害怕失败，所以不敢尝试任何新事物"},
        {"from": "gpt", "value": "害怕失败是人之常情，因为我们都希望避免痛苦和失望。但是你知道吗？很多时候，我们害怕的并不是失败本身，而是失败后别人的评价，或者对自己的否定。\n\n或许我们可以重新定义'失败'：它不是终点，而是通往成功路上的反馈信息。每一次'失败'都在告诉你，'下次可以试试别的方法'。"}
    ]},
    {"conversations": [
        {"from": "human", "value": "工作压力太大了，领导骂我，同事排挤我"},
        {"from": "gpt", "value": "在工作中被批评和排挤，这确实让人非常委屈和难受。你的感受是完全合理的，这种情况下感到沮丧、愤怒都是正常的反应。\n\n你愿意告诉我具体发生了什么吗？有时候把心里的委屈说出来，就已经在慢慢疗愈了。同时，我们也可以一起想想如何在这种环境中保护好自己的心理健康。"}
    ]},
]

sft_samples.extend(custom_samples)
print(f"添加自定义样本后，总样本数: {len(sft_samples)}")

# 保存为 JSONL
SFT_PATH = os.path.join(DATA_DIR, "emotional_sft.jsonl")
with open(SFT_PATH, "w", encoding="utf-8") as f:
    for sample in sft_samples:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")
print(f"已保存至: {SFT_PATH}")

In [ ]:
# @title 第六步：加载模型和 Tokenizer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"加载模型: {MODEL_NAME}")

# 4bit 量化配置 (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

model.gradient_checkpointing_enable()
model.config.use_cache = False

print(f"模型加载完成，显存占用: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
# @title 第七步：配置 LoRA 适配器
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 准备 4bit 模型进行训练
model = prepare_model_for_kbit_training(model)

# 配置 LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)

# 打印可训练参数
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数: {total_params:,}")
print(f"可训练参数: {trainable_params:,}")
print(f"训练比例: {trainable_params / total_params * 100:.2f}%")

In [ ]:
# @title 第八步：准备训练数据
from datasets import load_dataset, Dataset

def format_example(example):
    conversations = example["conversations"]
    messages = []
    for msg in conversations:
        role = "user" if msg["from"] == "human" else "assistant"
        messages.append({"role": role, "content": msg["value"]})
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokenized = tokenizer(text, max_length=MAX_LENGTH, truncation=True, padding="max_length", return_tensors="pt")
    
    input_ids = tokenized["input_ids"][0]
    attention_mask = tokenized["attention_mask"][0]
    
    # 构建 labels
    labels = input_ids.clone()
    text_decoded = tokenizer.decode(input_ids, skip_special_tokens=False)
    
    # 找到最后一个 assistant 回复的起始位置
    assistant_marker = "assistant\\n"
    last_pos = text_decoded.rfind(assistant_marker)
    if last_pos != -1:
        assistant_start = len(tokenizer.encode(text_decoded[:last_pos + len(assistant_marker)], add_special_tokens=False))
        labels[:assistant_start] = -100
    else:
        labels[:] = -100  # 如果找不到，设置全为 -100
        # 至少让最后几个 token 参与训练
        labels[-50:] = input_ids[-50:]
    
    return {
        "input_ids": input_ids.tolist(),
        "labels": labels.tolist(),
        "attention_mask": attention_mask.tolist(),
    }

# 加载 SFT 数据
dataset = load_dataset("json", data_files=SFT_PATH)
dataset = dataset["train"]
print(f"数据集大小: {len(dataset)}")

# 处理数据
print("处理数据...")
processed_dataset = dataset.map(format_example, remove_columns=dataset.column_names, desc="Processing")

# 过滤空样本
processed_dataset = processed_dataset.filter(lambda x: len(x["input_ids"]) > 0)
print(f"处理后样本数: {len(processed_dataset)}")

# 划分
split = processed_dataset.train_test_split(test_size=0.05, shuffle=True, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]
print(f"训练集: {len(train_dataset)}, 验证集: {len(eval_dataset)}")

In [ ]:
# @title 第九步：开始训练
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# 数据整理器
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
)

# 训练参数
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    eval_steps=200,
    evaluation_strategy="steps",
    save_total_limit=3,
    load_best_model_at_end=True,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    remove_unused_columns=False,
)

# 创建 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# 开始训练
print("开始训练...")
train_result = trainer.train()

# 保存模型
final_dir = os.path.join(OUTPUT_DIR, "final_model")
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

print(f"\n训练完成！模型保存至: {final_dir}")

In [ ]:
# @title 第十步：测试模型效果
import torch

def generate_response(user_input, max_new_tokens=256, temperature=0.7):
    messages = [{"role": "user", "content": user_input}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()

# 测试用例
test_cases = [
    "我今天心情不好，工作压力太大了",
    "我觉得自己什么都做不好，没有自信",
    "我失恋了，很难过",
    "我觉得没有人理解我",
    "我害怕失败，不敢尝试新事物",
]

print("=" * 60)
print("模型测试 - 情绪价值能力评估")
print("=" * 60)

for case in test_cases:
    print(f"\n{'─' * 50}")
    print(f"用户: {case}")
    response = generate_response(case)
    print(f"\n助手: {response}")

print(f"\n{'=' * 60}")

In [ ]:
# @title 第十一步：下载模型 (从 Google Drive)
if USE_DRIVE:
    print(f"模型已保存在 Google Drive: {OUTPUT_DIR}")
    print("你可以从 Google Drive 下载模型文件")
else:
    # 打包下载
    import shutil
    zip_path = "/content/emotional_qwen_model.zip"
    shutil.make_archive(
        zip_path.replace(".zip", ""),
        'zip',
        os.path.join(OUTPUT_DIR, "final_model")
    )
    
    from google.colab import files
    files.download(zip_path)
    print("模型已打包下载")

## 🎉 训练完成

模型已成功微调并下载！你现在可以：

1. **本地部署**：使用下载的模型在本地运行推理服务
2. **继续训练**：在 Google Colab 中继续训练更多 epoch
3. **分享**：将模型分享给他人使用

## 💡 提示

- 使用 T4 GPU 训练约需 30-60 分钟
- 使用 A100 GPU 训练约需 10-20 分钟
- 建议训练 3-5 个 epoch 以获得更好的情绪价值能力
- 可以增加更多高质量的情绪对话数据来提升效果